# RLHF에서 DPO로 — 이론

> SFT는 "따라해라", DPO는 "이것이 저것보다 낫다" — 모델에게 **판단력**을 심는다

Phase 4에서 SFT로 모델이 지시를 따르게 만들었다.  
이제 같은 질문에 대해 **좋은 답과 나쁜 답을 구분**하는 능력을 추가한다.

In [ ]:
# === 환경 설치 ===
!pip install matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("Phase 6: 모델 정렬 — DPO 이론")

---
## 1. SFT의 한계

SFT는 "이렇게 답해라"만 가르친다. **여러 가능한 답 중 어떤 것이 더 나은지**는 배우지 못한다.

| 학습 방식 | 가르치는 것 | 한계 |
|----------|-----------|------|
| SFT | "이렇게 답해라" | 여러 답 중 우열 구분 불가 |
| RLHF/DPO | "이 답이 저 답보다 낫다" | 상대적 선호도 학습 |

In [ ]:
# === SFT의 한계 시연 ===

print("질문: '파이썬 정렬 방법은?'")
print("\n" + "=" * 50)
print("응답 A (고품질):")
print("  sorted()는 새 리스트를 반환하고, .sort()는 제자리 정렬합니다.")
print("  예시: sorted([3,1,2]) → [1,2,3]")
print("  key 파라미터로 정렬 기준을 커스텀할 수 있습니다.")

print("\n응답 B (저품질):")
print("  sort 쓰면 됩니다.")

print("\n" + "=" * 50)
print("SFT의 문제: 두 응답 모두 '정답'으로 취급")
print("DPO의 해결: A > B라는 선호도를 학습")

---
## 2. RLHF vs DPO

RLHF는 3단계 파이프라인으로 복잡하다. DPO는 이를 **단일 손실함수**로 단순화했다.

```
RLHF: SFT모델 → Reward Model 학습 → PPO 강화학습 (3단계)
DPO:  SFT모델 → 직접 최적화                      (1단계)
```

In [ ]:
# === RLHF vs DPO 비교 시각화 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RLHF 파이프라인
rlhf_stages = ['SFT 모델', 'Reward Model\n학습', 'PPO\n강화학습', '최종 모델']
rlhf_complexity = [2, 4, 5, 1]  # 난이도
colors_rlhf = ['#51cf66', '#ff6b6b', '#ff6b6b', '#51cf66']
axes[0].barh(rlhf_stages, rlhf_complexity, color=colors_rlhf, edgecolor='black')
axes[0].set_xlabel('구현 복잡도')
axes[0].set_title('RLHF (PPO) — 3단계')
axes[0].set_xlim(0, 6)
for i, v in enumerate(rlhf_complexity):
    axes[0].text(v + 0.1, i, f'{v}/5', va='center')

# DPO 파이프라인
dpo_stages = ['SFT 모델', 'DPO 직접\n최적화', '최종 모델']
dpo_complexity = [2, 2, 1]
colors_dpo = ['#51cf66', '#339af0', '#51cf66']
axes[1].barh(dpo_stages, dpo_complexity, color=colors_dpo, edgecolor='black')
axes[1].set_xlabel('구현 복잡도')
axes[1].set_title('DPO — 1단계')
axes[1].set_xlim(0, 6)
for i, v in enumerate(dpo_complexity):
    axes[1].text(v + 0.1, i, f'{v}/5', va='center')

plt.suptitle('RLHF vs DPO: 복잡도 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 비교표
print("\n상세 비교:")
print(f"{'항목':<20} {'RLHF (PPO)':<25} {'DPO':<25}")
print("-" * 70)
comparisons = [
    ("필요 모델 수", "3 (SFT+Reward+Policy)", "2 (Reference+Policy)"),
    ("학습 안정성", "불안정 (HP 민감)", "안정 (분류 손실함수)"),
    ("구현 복잡도", "높음", "낮음"),
    ("성능", "이론상 우위", "실전 동등"),
    ("실무 채택률", "감소 추세", "증가 추세"),
]
for item, rlhf, dpo in comparisons:
    print(f"{item:<20} {rlhf:<25} {dpo:<25}")

---
## 3. DPO 손실 함수

$$L_{DPO} = -\log \sigma(\beta \times (\log \frac{\pi(w)}{\pi_{ref}(w)} - \log \frac{\pi(l)}{\pi_{ref}(l)}))$$

| 기호 | 의미 |
|------|------|
| π | policy model (학습 중) |
| π_ref | reference model (동결) |
| w | chosen (선호 응답) |
| l | rejected (비선호 응답) |
| β | KL 제약 강도 |
| σ | sigmoid 함수 |

In [ ]:
# === DPO 손실 함수 시각화 ===

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dpo_loss(log_ratio_w, log_ratio_l, beta=0.1):
    """DPO Loss 계산"""
    return -np.log(sigmoid(beta * (log_ratio_w - log_ratio_l)))

# log_ratio = log(π/π_ref): policy가 reference 대비 얼마나 변했는가
# chosen의 log_ratio가 높고, rejected의 log_ratio가 낮으면 → Loss 감소

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) chosen log_ratio 변화에 따른 Loss
log_ratio_w = np.linspace(-5, 5, 100)
log_ratio_l_fixed = 0  # rejected는 변화 없음
loss_1 = [dpo_loss(lw, log_ratio_l_fixed, beta=0.1) for lw in log_ratio_w]
axes[0].plot(log_ratio_w, loss_1, 'b-', linewidth=2)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('log(π_chosen / π_ref_chosen)')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('chosen 확률 ↑ → Loss ↓')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('chosen 확률이 높아질수록\nLoss 감소', xy=(3, 0.2), fontsize=10)

# 2) rejected log_ratio 변화에 따른 Loss
log_ratio_w_fixed = 0  # chosen은 변화 없음
log_ratio_l = np.linspace(-5, 5, 100)
loss_2 = [dpo_loss(log_ratio_w_fixed, ll, beta=0.1) for ll in log_ratio_l]
axes[1].plot(log_ratio_l, loss_2, 'r-', linewidth=2)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('log(π_rejected / π_ref_rejected)')
axes[1].set_ylabel('DPO Loss')
axes[1].set_title('rejected 확률 ↓ → Loss ↓')
axes[1].grid(True, alpha=0.3)
axes[1].annotate('rejected 확률이 낮아질수록\nLoss 감소', xy=(-4.5, 0.2), fontsize=10)

# 3) margin (chosen - rejected) 효과
margins = np.linspace(-5, 5, 100)
for beta in [0.05, 0.1, 0.3]:
    loss_3 = [-np.log(sigmoid(beta * m)) for m in margins]
    axes[2].plot(margins, loss_3, linewidth=2, label=f'β={beta}')
axes[2].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('log_ratio_chosen - log_ratio_rejected')
axes[2].set_ylabel('DPO Loss')
axes[2].set_title('Margin 효과 (β별)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('DPO 손실 함수 직관', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("핵심:")
print("  1. chosen에 높은 확률 → Loss 감소")
print("  2. rejected에 낮은 확률 → Loss 감소")
print("  3. π_ref로 나누기 → 원본에서 너무 벗어나지 않도록 제약")

---
## 4. β (beta) 하이퍼파라미터

reference model에서 얼마나 벗어나도 허용하는지를 결정한다.

| β | 효과 | 용도 |
|---|------|------|
| 0.05 | 공격적, 빠른 정렬 | 과적합 위험 |
| **0.1** | **표준** | **대부분의 시작점** |
| 0.3~0.5 | 보수적, 안전 | 느린 수렴 |

In [ ]:
# === β 효과 시각화 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) β별 Loss 곡선
margins = np.linspace(-10, 10, 200)
betas = [0.01, 0.05, 0.1, 0.3, 0.5]
colors = ['#ff6b6b', '#ffa07a', '#339af0', '#51cf66', '#2d8a4e']

for beta, color in zip(betas, colors):
    loss = [-np.log(sigmoid(beta * m)) for m in margins]
    axes[0].plot(margins, loss, color=color, linewidth=2, label=f'β={beta}')

axes[0].set_xlabel('chosen - rejected margin')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('β별 Loss 민감도')
axes[0].legend()
axes[0].set_ylim(0, 3)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)

# 2) β별 gradient 크기
for beta, color in zip(betas, colors):
    grad = [beta * (1 - sigmoid(beta * m)) for m in margins]
    axes[1].plot(margins, grad, color=color, linewidth=2, label=f'β={beta}')

axes[1].set_xlabel('chosen - rejected margin')
axes[1].set_ylabel('Gradient 크기')
axes[1].set_title('β별 학습 신호 강도')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)

plt.suptitle('β 하이퍼파라미터 효과', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("해석:")
print("  β 작음 (0.05): Loss 곡선이 완만 → 큰 변화 허용 → 빠른 정렬, 과적합 위험")
print("  β 큼 (0.5):   Loss 곡선이 급격 → 작은 변화만 허용 → 안전한 정렬, 느린 수렴")
print("  β = 0.1:      표준 시작점")

---
## 5. Reference Model의 역할

Reference model = SFT 완료 직후의 모델 (동결).  
학습 중 policy model이 reference에서 너무 멀어지면 penalty → **KL divergence 제약**.

In [ ]:
# === Reference Model KL 제약 시뮬레이션 ===

np.random.seed(42)

# 시뮬레이션: 학습 스텝에 따른 policy 변화
steps = np.arange(0, 100)

# KL divergence 시뮬레이션
kl_with_constraint = 0.5 * (1 - np.exp(-0.03 * steps))  # β 제약 있을 때
kl_without = 0.02 * steps  # 제약 없을 때 (선형 발산)

# Loss 시뮬레이션
loss_with = 0.7 * np.exp(-0.05 * steps) + 0.1 + np.random.normal(0, 0.02, len(steps))
loss_without = 0.7 * np.exp(-0.08 * steps) + 0.05 + np.random.normal(0, 0.03, len(steps))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) KL Divergence
axes[0].plot(steps, kl_with_constraint, 'b-', linewidth=2, label='DPO (β=0.1, 제약 있음)')
axes[0].plot(steps, kl_without, 'r-', linewidth=2, label='제약 없음 (발산)')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='안전 상한')
axes[0].set_xlabel('학습 스텝')
axes[0].set_ylabel('KL(π || π_ref)')
axes[0].set_title('Reference Model이 발산을 방지')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2) Loss
axes[1].plot(steps, loss_with, 'b-', alpha=0.7, label='DPO (안정적 수렴)')
axes[1].plot(steps, loss_without, 'r-', alpha=0.7, label='제약 없음 (과적합 후 발산 위험)')
axes[1].set_xlabel('학습 스텝')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss 수렴 비교')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Reference Model의 KL 제약 효과', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("TRL DPOTrainer에서:")
print("  ref_model=None → 학습 시작 시 policy의 복사본을 자동으로 reference로 사용")
print("  → 별도로 reference model을 관리할 필요 없음")

---
## 6. SFT → DPO 전체 파이프라인

```
Phase 4: SFT 모델 (지시 따르기)
    ├── 동결 복사 → Reference Model
    └── 학습 대상 → Policy Model
         ↓
Phase 6-02: Preference 데이터 준비 (prompt + chosen + rejected)
         ↓
Phase 6-03: DPO 학습 (DPOTrainer)
         ↓
Phase 6-04: GRPO + 다음 단계
```

In [ ]:
# === 파이프라인 요약 시각화 ===

fig, ax = plt.subplots(figsize=(12, 4))

phases = ['Phase 4\nSFT', 'Preference\n데이터 준비', 'DPO\n학습', 'GRPO\n(선택)']
phase_x = [0, 1, 2, 3]
colors = ['#51cf66', '#ffa07a', '#339af0', '#c4c4c4']

for i, (x, label, color) in enumerate(zip(phase_x, phases, colors)):
    circle = plt.Circle((x, 0), 0.3, color=color, ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(x, 0, label, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(phases) - 1:
        ax.annotate('', xy=(x + 0.4, 0), xytext=(x + 0.6, 0),
                    arrowprops=dict(arrowstyle='->', lw=2))

ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.6, 0.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('모델 정렬 파이프라인', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n핵심 포인트:")
print("  1. SFT가 '무엇을 답할지' 가르치고")
print("  2. DPO가 '어떻게 더 잘 답할지' 가르친다")
print("  3. SFT 품질이 DPO의 상한선을 결정한다")
print("  4. Reference model이 원본에서 너무 벗어나지 않도록 제약한다")

---
## 정리

| 개념 | 핵심 |
|------|------|
| **SFT 한계** | 여러 답 중 우열 구분 불가 |
| **RLHF** | 3단계 (Reward Model + PPO), 복잡 |
| **DPO** | 단일 손실함수, 안정적, 실전 동등 성능 |
| **β** | KL 제약 강도 (0.1이 표준) |
| **Reference Model** | 동결된 SFT 모델, 발산 방지 |

### 핵심 원칙

> **DPO의 본질은 "정답을 가르치는 것"이 아니라 "비교하는 능력을 가르치는 것"이다.**  
> 절대적으로 완벽한 답은 없어도, 상대적으로 나은 답은 항상 있다.

### 다음: 02_Preference_데이터_준비.ipynb

DPO 학습에 필요한 Preference 데이터(prompt + chosen + rejected)를 준비한다.